<div class="alert alert-danger" role="alert">
<h1 align="center"><font size = 14>Lid-Driven Cavity (PINN)</font></h1>
<h4 align="center">Aug, 16_2026<h4>
<h3 align="center">Armin Amani</h3>

<div class="alert alert-danger" role="alert">
📤 Import Libraries

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pyDOE import lhs
import time

<div class="alert alert-danger" role="alert"> 
🔎 Set device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

<div class="alert alert-danger" role="alert"> 
🔎 seeds for reproducibility

In [ ]:
seed = 1234
torch.manual_seed(seed)
np.random.seed(seed)

<div class="alert alert-danger" role="alert"> 
🔎 Problem parameters

In [ ]:
x_min, x_max = 0.0, 1.0
y_min, y_max = 0.0, 1.0
U_lid = 1.0                                 # Lid velocity
nu = 0.01                                   # Kinematic viscosity
Re = U_lid * (x_max - x_min) / nu           # Reynolds number

<div class="alert alert-danger" role="alert"> 
🔎 Domain boundaries

In [ ]:
ub = torch.tensor([x_max, y_max], dtype=torch.float32).to(device)
lb = torch.tensor([x_min, y_min], dtype=torch.float32).to(device)

<div class="alert alert-danger" role="alert"> 
🔎 Number of points

In [ ]:
N_f = 10000  # Collocation points
N_b = 2000   # Boundary points (500 per side)

def getData():
    # Collocation points (requires_grad will be set during training)
    xy_f = lb + (ub - lb) * torch.tensor(lhs(2, N_f), dtype=torch.float32).to(device)
    
    # Boundary points
    # Left wall (x=0)
    x_left = torch.zeros((N_b//4, 1), dtype=torch.float32).to(device)
    y_left = torch.tensor(y_min + (y_max - y_min) * lhs(1, N_b//4), dtype=torch.float32).to(device)
    xy_left = torch.cat([x_left, y_left], dim=1)
    
    # Right wall (x=1)
    x_right = torch.ones((N_b//4, 1), dtype=torch.float32).to(device)
    y_right = torch.tensor(y_min + (y_max - y_min) * lhs(1, N_b//4), dtype=torch.float32).to(device)
    xy_right = torch.cat([x_right, y_right], dim=1)
    
    # Bottom wall (y=0)
    x_bottom = torch.tensor(x_min + (x_max - x_min) * lhs(1, N_b//4), dtype=torch.float32).to(device)
    y_bottom = torch.zeros((N_b//4, 1), dtype=torch.float32).to(device)
    xy_bottom = torch.cat([x_bottom, y_bottom], dim=1)
    
    # Top lid (y=1)
    x_top = torch.tensor(x_min + (x_max - x_min) * lhs(1, N_b//4), dtype=torch.float32).to(device)
    y_top = torch.ones((N_b//4, 1), dtype=torch.float32).to(device)
    xy_top = torch.cat([x_top, y_top], dim=1)
    
    # Combine all boundary points
    xy_b = torch.cat([xy_left, xy_right, xy_bottom, xy_top], dim=0)
    
    # Boundary conditions (u, v)
    u_top = U_lid * torch.ones((N_b//4, 1), dtype=torch.float32).to(device)
    v_top = torch.zeros((N_b//4, 1), dtype=torch.float32).to(device)
    
    u_side = torch.zeros((3*(N_b//4), 1), dtype=torch.float32).to(device)
    v_side = torch.zeros((3*(N_b//4), 1), dtype=torch.float32).to(device)
    
    u_b = torch.cat([u_side, u_top], dim=0)
    v_b = torch.cat([v_side, v_top], dim=0)
    
    return xy_f, xy_b, u_b, v_b

xy_f, xy_b, u_b, v_b = getData()

<div class="alert alert-danger" role="alert"> 
🔎 Setting Function of Plot

In [ ]:
def plot_results(X, Y, U, V, P, losses):
    """Plot velocity fields, streamlines, pressure contours and loss curves"""
    plt.figure(figsize=(18, 12))
    
    # Velocity magnitude
    plt.subplot(2, 3, 1)
    vel_mag = np.sqrt(U**2 + V**2)
    plt.pcolormesh(X, Y, vel_mag, cmap='jet', shading='auto')
    plt.colorbar(label='Velocity Magnitude')
    plt.streamplot(X, Y, U, V, color='k', density=1.5)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Velocity Magnitude with Streamlines')
    
    # U-velocity
    plt.subplot(2, 3, 2)
    plt.pcolormesh(X, Y, U, cmap='jet', shading='auto')
    plt.colorbar(label='u velocity')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Horizontal Velocity (u)')
    
    # V-velocity
    plt.subplot(2, 3, 3)
    plt.pcolormesh(X, Y, V, cmap='jet', shading='auto')
    plt.colorbar(label='v velocity')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Vertical Velocity (v)')
    
    # Pressure
    plt.subplot(2, 3, 4)
    plt.pcolormesh(X, Y, P, cmap='jet', shading='auto')
    plt.colorbar(label='Pressure')
    plt.contour(X, Y, P, 20, colors='k', linewidths=0.5)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Pressure Contours')
    
    # Loss curves
    plt.subplot(2, 3, 5)
    plt.plot(losses['bc'], label='BC Loss')
    plt.plot(losses['pde'], label='PDE Loss')
    plt.yscale('log')
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training Loss History')
    
    plt.tight_layout()
    plt.savefig('lid_driven_cavity_results.png')
    plt.close()



def weights_init(m):
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.xavier_normal_(m.weight.data)
        torch.nn.init.zeros_(m.bias.data)

<div class="alert alert-danger" role="alert"> 
🔎 Layer Class

In [ ]:
class Layer(torch.nn.Module):
    def __init__(self, n_in, n_out, activation):
        super().__init__()
        self.layer = torch.nn.Linear(n_in, n_out)
        self.activation = activation
    
    def forward(self, x):
        x = self.layer(x)
        if self.activation:
            x = self.activation(x)
        return x

<div class="alert alert-danger" role="alert"> 
🔎 DNN Class

In [ ]:
class DNN(torch.nn.Module):
    def __init__(self, dim_in=2, dim_out=3, n_layer=8, n_node=30, ub=ub, lb=lb, activation=torch.nn.Tanh()):
        super().__init__()
        self.net = torch.nn.ModuleList()
        self.net.append(Layer(dim_in, n_node, activation))
        for _ in range(n_layer):
            self.net.append(Layer(n_node, n_node, activation))
        self.net.append(Layer(n_node, dim_out, activation=None))
        self.ub = torch.tensor(ub, dtype=torch.float32).to(device)
        self.lb = torch.tensor(lb, dtype=torch.float32).to(device)
        self.net.apply(weights_init)
    
    def forward(self, x):
        x = (x - self.lb) / (self.ub - self.lb)  # Normalize input
        out = x
        for layer in self.net:
            out = layer(out)
        return out

<div class="alert alert-danger" role="alert"> 
🔎 PINN Class

In [ ]:
class PINN:
    def __init__(self):
        self.net = DNN(dim_in=2, dim_out=3, n_layer=8, n_node=30, ub=ub, lb=lb).to(device)
        self.lbfgs = torch.optim.LBFGS(
            self.net.parameters(),
            lr=1.0,
            max_iter=10000,
            max_eval=10000,
            tolerance_grad=1e-10,
            tolerance_change=np.finfo(float).eps,
            history_size=100,
            line_search_fn="strong_wolfe",
        )
        self.adam = torch.optim.Adam(self.net.parameters(), lr=0.001)
        self.losses = {"bc": [], "pde": [], "continuity": [], "momentum_x": [], "momentum_y": []}
        self.iter = 0
    
    def predict(self, x):
        out = self.net(x)
        u = out[:, 0:1]
        v = out[:, 1:2]
        p = out[:, 2:3]
        return u, v, p
    
    def pde_loss(self, x):
        x = x.clone()
        x.requires_grad = True
        
        u, v, p = self.predict(x)
        
        # First derivatives
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0][:, 0:1]
        u_y = torch.autograd.grad(u.sum(), x, create_graph=True)[0][:, 1:2]
        v_x = torch.autograd.grad(v.sum(), x, create_graph=True)[0][:, 0:1]
        v_y = torch.autograd.grad(v.sum(), x, create_graph=True)[0][:, 1:2]
        
        p_x = torch.autograd.grad(p.sum(), x, create_graph=True)[0][:, 0:1]
        p_y = torch.autograd.grad(p.sum(), x, create_graph=True)[0][:, 1:2]
        
        # Second derivatives
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0][:, 0:1]
        u_yy = torch.autograd.grad(u_y.sum(), x, create_graph=True)[0][:, 1:2]
        v_xx = torch.autograd.grad(v_x.sum(), x, create_graph=True)[0][:, 0:1]
        v_yy = torch.autograd.grad(v_y.sum(), x, create_graph=True)[0][:, 1:2]
        
        # Continuity equation
        continuity = u_x + v_y
        
        # Momentum equations
        momentum_x = u*u_x + v*u_y + p_x - (1/Re)*(u_xx + u_yy)
        momentum_y = u*v_x + v*v_y + p_y - (1/Re)*(v_xx + v_yy)
        
        # Individual losses
        mse_continuity = torch.mean(continuity**2)
        mse_momentum_x = torch.mean(momentum_x**2)
        mse_momentum_y = torch.mean(momentum_y**2)
        mse_pde = mse_continuity + mse_momentum_x + mse_momentum_y
        
        # Store individual losses
        self.losses["continuity"].append(mse_continuity.detach().cpu().item())
        self.losses["momentum_x"].append(mse_momentum_x.detach().cpu().item())
        self.losses["momentum_y"].append(mse_momentum_y.detach().cpu().item())
        
        return mse_pde
    
    def bc_loss(self, xy_b, u_b, v_b):
        u_pred, v_pred, _ = self.predict(xy_b)
        mse_bc = torch.mean(torch.square(u_pred - u_b)) + torch.mean(torch.square(v_pred - v_b))
        return mse_bc
    
    def closure(self):
        self.lbfgs.zero_grad()
        self.adam.zero_grad()
        mse_pde = self.pde_loss(xy_f)
        mse_bc = self.bc_loss(xy_b, u_b, v_b)
        loss = mse_pde + mse_bc
        loss.backward()
        
        self.losses["bc"].append(mse_bc.detach().cpu().item())
        self.losses["pde"].append(mse_pde.detach().cpu().item())
        self.iter += 1
        
        print(f"\rIter: {self.iter} Loss: {loss.item():.3e} BC: {mse_bc.item():.3e} PDE: {mse_pde.item():.3e}", end="")
        if self.iter % 100 == 0:
            print("")
        return loss

<div class="alert alert-danger" role="alert"> 
🔎 Train and evaluate the model

In [ ]:
if __name__ == "__main__":
    pinn = PINN()
    start_time = time.time()
    
    # Train with Adam
    for i in range(1000):
        pinn.closure()
        pinn.adam.step()
    
    # Train with L-BFGS
    pinn.lbfgs.step(pinn.closure)
    
    print(f"\nTraining completed in {(time.time() - start_time)/60:.2f} minutes")
    
    # Create prediction grid
    nx, ny = 100, 100
    x = torch.linspace(x_min, x_max, nx, dtype=torch.float32).to(device)
    y = torch.linspace(y_min, y_max, ny, dtype=torch.float32).to(device)
    X, Y = torch.meshgrid(x, y, indexing='xy')
    xy_grid = torch.stack([X.flatten(), Y.flatten()], dim=1)
    
    # Predict solution
    with torch.no_grad():
        u_pred, v_pred, p_pred = pinn.predict(xy_grid)
        U = u_pred.reshape(nx, ny).cpu().numpy()
        V = v_pred.reshape(nx, ny).cpu().numpy()
        P = p_pred.reshape(nx, ny).cpu().numpy()
        X = X.cpu().numpy()
        Y = Y.cpu().numpy()
    
    # Plot results
    plot_results(X, Y, U, V, P, pinn.losses)
    
    # Save model
    torch.save(pinn.net.state_dict(), "lid_driven_cavity_pinn.pt")